# DefectVision AI - Phase 2: Anomaly Detection
## Train PatchCore on Normal/Good Images Only

**Phase 2 is fundamentally different from Phase 1:**
- Phase 1 (YOLO) needs labeled defect images with bounding boxes
- Phase 2 (PatchCore) only needs **normal/good images** -- no labels!
- It learns what "normal" looks like, then flags anything different
- Outputs a **heatmap** showing exactly where the anomaly is

This is how real factories work -- defects are rare and you can't label every possible type.

We use **Anomalib** (by Intel) which implements PatchCore, PaDiM, and many other methods.

**Supports: Google Colab, Kaggle, and local training.**
1. **Kaggle** (recommended): Use GPU P100 or T4x2 accelerator
2. **Colab**: Go to Runtime > Change runtime type > GPU (T4)
3. **Local**: Requires NVIDIA GPU with CUDA

## Step 1: Install Dependencies

In [ ]:
import os, sys

def detect_environment():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "kaggle"
    try:
        import google.colab
        return "colab"
    except ImportError:
        pass
    return "local"

ENV = detect_environment()
print(f"Detected environment: {ENV.upper()}")

if ENV in ("kaggle", "colab"):
    os.system("pip install anomalib -q")
    print("Installed anomalib")
else:
    print("Local mode: run 'pip install anomalib' if not already installed")

## Step 2: Check GPU & Configure

In [ ]:
import torch

print(f"Environment:     {ENV.upper()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    GPU_NAME = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    VRAM_GB = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
    print(f"GPU:             {GPU_NAME}")
    print(f"VRAM:            {VRAM_GB:.1f} GB")
    BATCH_SIZE = 32 if VRAM_GB >= 8 else 16
else:
    BATCH_SIZE = 8
    print("WARNING: No GPU detected. PatchCore will be slow on CPU.")
    if ENV == "kaggle":
        print("  -> Go to Settings > Accelerator > GPU P100")
    elif ENV == "colab":
        print("  -> Go to Runtime > Change runtime type > GPU (T4)")

print(f"\nBatch size:  {BATCH_SIZE}")

## Step 3: Download MVTec AD Dataset

MVTec Anomaly Detection (MVTec AD) is the gold-standard benchmark for anomaly detection.
It has 15 categories of objects/textures, each with:
- A training set of only **good/normal** images
- A test set with both good and defective images
- Pixel-level ground truth masks for defects

We'll use the **metal_nut** category (relevant to our metal domain).
Anomalib handles the download automatically.

In [ ]:
from pathlib import Path
import os, tarfile, urllib.request

CATEGORY = "metal_nut"
IMAGE_SIZE = (256, 256)
DATASET_ROOT = Path("./datasets/MVTecAD")

MVTEC_CATEGORY_URLS = {
    "metal_nut": "https://www.mydrive.ch/shares/38536/3830184030e49fe74747669442f0f283/download/420937637-1629959294/metal_nut.tar.xz",
}

if not (DATASET_ROOT / CATEGORY / "train").exists():
    DATASET_ROOT.mkdir(parents=True, exist_ok=True)
    downloaded = False

    if CATEGORY in MVTEC_CATEGORY_URLS:
        url = MVTEC_CATEGORY_URLS[CATEGORY]
        tar_path = DATASET_ROOT / f"{CATEGORY}.tar.xz"
        try:
            print(f"Downloading '{CATEGORY}' from MVTec official mirror (~157 MB)...")
            urllib.request.urlretrieve(url, tar_path)
            print("Extracting...")
            with tarfile.open(tar_path, "r:xz") as tf:
                tf.extractall(DATASET_ROOT)
            tar_path.unlink()
            downloaded = True
            print(f"Dataset ready at {DATASET_ROOT / CATEGORY}")
        except Exception as e:
            print(f"Direct download failed: {e}")
            if tar_path.exists():
                tar_path.unlink()

    if not downloaded and ENV in ("kaggle", "colab"):
        try:
            print("Trying kagglehub fallback...")
            import kagglehub
            dl_path = Path(kagglehub.dataset_download("ipythonx/mvtec-ad"))
            candidates = sorted(dl_path.rglob(CATEGORY), key=lambda p: len(p.parts))
            if candidates:
                src = candidates[0].resolve()
                target = DATASET_ROOT / CATEGORY
                if not target.exists():
                    os.symlink(src, target)
                downloaded = True
                print(f"Linked: {src} -> {target}")
        except Exception as e:
            print(f"kagglehub fallback also failed: {e}")

    if not downloaded:
        raise FileNotFoundError(
            f"Could not download MVTec AD. Please download '{CATEGORY}' manually from:\n"
            f"  https://www.mvtec.com/company/research/datasets/mvtec-ad/downloads\n"
            f"  and extract into {DATASET_ROOT}/"
        )
else:
    print(f"Dataset already present at {DATASET_ROOT / CATEGORY}")

In [ ]:
from anomalib.data import MVTecAD
from torchvision.transforms.v2 import Resize, Compose

augmentations = Compose([Resize(IMAGE_SIZE)])

datamodule = MVTecAD(
    root=str(DATASET_ROOT),
    category=CATEGORY,
    train_batch_size=BATCH_SIZE,
    eval_batch_size=BATCH_SIZE,
    num_workers=2 if ENV == "local" else 4,
    augmentations=augmentations,
)

datamodule.setup()

print(f"Category: {CATEGORY}")
print(f"Image size: {IMAGE_SIZE}")
print(f"Training samples (good only): {len(datamodule.train_data)}")
print(f"Test samples (good + defective): {len(datamodule.test_data)}")

## Step 4: Visualize Training Data (All Normal)

Notice: the training set contains ONLY normal/good images. No defects.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

train_loader = datamodule.train_dataloader()
batch = next(iter(train_loader))

images = batch.image if hasattr(batch, 'image') else batch["image"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flatten()):
    if i < len(images):
        img = images[i].permute(1, 2, 0).cpu().numpy()
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title("NORMAL (training)", fontsize=9, color="green")
    ax.axis('off')

plt.suptitle(f'Training Data - {CATEGORY} (all normal/good)', fontsize=14)
plt.tight_layout()
plt.show()

## Step 5: Train PatchCore Model

**PatchCore** works by:
1. Extracting feature patches from normal images using a pretrained CNN backbone
2. Building a memory bank of these "normal" features
3. At test time, comparing new image patches against the memory bank
4. High distance = anomaly

Training is fast (~5-10 minutes) because there's no gradient descent -- it just builds the memory bank.

In [ ]:
from anomalib.models import Patchcore
from anomalib.engine import Engine
from functools import partial
from tqdm import tqdm
import anomalib.models.components.sampling.k_center_greedy as _kcg

# Disable tqdm in coreset selection to prevent rich/Jupyter/IPython recursion crash on Kaggle
_kcg.tqdm = partial(tqdm, disable=True)

model = Patchcore(
    backbone="wide_resnet50_2",
    layers=("layer2", "layer3"),
    pre_trained=True,
    coreset_sampling_ratio=0.1,
    num_neighbors=9,
)

engine = Engine(
    max_epochs=1,
    default_root_dir="./anomaly_results",
)

print("Training PatchCore (building memory bank from normal images)...")
print("Coreset selection in progress (this takes ~1-2 min, progress bar disabled for Kaggle compatibility)...")
engine.fit(datamodule=datamodule, model=model)
print("Training complete!")

## Step 6: Evaluate and Visualize Results

Test on images that include both normal and defective samples.
The model generates:
- **Anomaly score** (0-1, higher = more anomalous)
- **Anomaly heatmap** showing where defects are

In [ ]:
test_results = engine.test(datamodule=datamodule, model=model)
print(f"\n=== Anomaly Detection Results ({CATEGORY}) ===")
for key, value in test_results[0].items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
batch_preds = engine.predict(datamodule=datamodule, model=model)

# Flatten batched predictions into individual samples
samples = []
for batch in batch_preds:
    batch_size = batch.image.shape[0]
    for j in range(batch_size):
        samples.append({
            "image": batch.image[j],
            "anomaly_map": batch.anomaly_map[j] if batch.anomaly_map is not None else None,
            "gt_mask": batch.gt_mask[j] if batch.gt_mask is not None else None,
            "pred_score": batch.pred_score[j] if batch.pred_score is not None else None,
            "pred_label": batch.pred_label[j] if batch.pred_label is not None else None,
        })

num_show = min(6, len(samples))
fig, axes = plt.subplots(num_show, 4, figsize=(20, 5 * num_show))
fig.suptitle(f'Anomaly Detection Results - {CATEGORY}', fontsize=16)

if num_show == 1:
    axes = [axes]

for i in range(num_show):
    s = samples[i]

    img = s["image"].permute(1, 2, 0).cpu().numpy()
    img = np.clip(img, 0, 1)
    axes[i][0].imshow(img)
    axes[i][0].set_title("Original", fontsize=10)

    if s["anomaly_map"] is not None:
        amap = s["anomaly_map"].squeeze().cpu().numpy()
        axes[i][1].imshow(amap, cmap="jet")
        axes[i][1].set_title("Anomaly Heatmap", fontsize=10)

    if s["gt_mask"] is not None:
        mask = s["gt_mask"].squeeze().cpu().numpy()
        axes[i][2].imshow(mask, cmap="gray")
        axes[i][2].set_title("Ground Truth", fontsize=10)

    score = float(s["pred_score"]) if s["pred_score"] is not None else 0.0
    is_anomaly = bool(s["pred_label"]) if s["pred_label"] is not None else False
    label = "ANOMALY" if is_anomaly else "NORMAL"
    color = "red" if is_anomaly else "green"
    axes[i][3].text(0.5, 0.5, f"{label}\nScore: {score:.3f}",
                    ha='center', va='center', fontsize=16, color=color,
                    transform=axes[i][3].transAxes)
    axes[i][3].set_title("Verdict", fontsize=10)

    for ax in axes[i]:
        ax.axis('off')

plt.tight_layout()
plt.show()

## Step 7: Export Model for Local Use

Export the trained model so we can use it in our Gradio app.

In [ ]:
import shutil, os
from pathlib import Path
from anomalib.deploy import ExportType

results_dir = Path("./anomaly_results")
exported = False

# Try OpenVINO export (install on-the-fly if needed)
try:
    import openvino
except ImportError:
    if ENV in ("kaggle", "colab"):
        print("Installing OpenVINO for export...")
        os.system("pip install openvino -q")

try:
    engine.export(model=model, export_type=ExportType.OPENVINO, input_size=IMAGE_SIZE)
    exported = True
    print("Exported as OpenVINO model")
except Exception as e:
    print(f"OpenVINO export failed: {e}")

# Fallback: export as ONNX
if not exported:
    try:
        engine.export(model=model, export_type=ExportType.ONNX, input_size=IMAGE_SIZE)
        exported = True
        print("Exported as ONNX model")
    except Exception as e:
        print(f"ONNX export also failed: {e}")

# The .ckpt checkpoint is always saved during training as fallback
print("\nAll model files found:")
for ext in ("*.pt", "*.ckpt", "*.bin", "*.xml", "*.onnx"):
    for path in results_dir.rglob(ext):
        print(f"  {path} ({path.stat().st_size / 1e6:.1f} MB)")

In [ ]:
import zipfile

zip_name = "anomaly_model.zip"
included_extensions = {".pt", ".ckpt", ".yaml", ".json", ".bin", ".xml", ".txt"}
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file_path in results_dir.rglob("*"):
        if file_path.is_file() and file_path.suffix in included_extensions:
            zipf.write(file_path, file_path.relative_to(results_dir))
            print(f"  Added: {file_path.relative_to(results_dir)}")

print(f"\nZipped model: {zip_name}")

try:
    from google.colab import files
    files.download(zip_name)
    print("\nDownload started! Unzip into models/anomaly_model/")
except ImportError:
    if ENV == "kaggle":
        print("\nKaggle: find anomaly_model.zip in the Output tab")
    else:
        print("\nLocal: unzip anomaly_model.zip into models/anomaly_model/")

## What Just Happened?

**Key takeaway for the hackathon presentation:**

1. We trained on ONLY normal/good images -- no defect labels needed
2. The model learned a "memory bank" of what normal metal nuts look like
3. At test time, it compares new images against this memory bank
4. Any region that doesn't match gets flagged with a heatmap
5. This works for **any** defect type -- even ones never seen before

This is why anomaly detection is more practical for manufacturing:
- Defects are rare (maybe 1 in 1000 items)
- You can't label every possible defect type
- New defect types appear over time
- You always have plenty of normal/good samples